# Exploración y preparación reproducible de Avance del Plan

## 1. Propósito y unidad de observación

Preparo las tablas físicas del plan sin alterar los originales. Identifico esquema, nivel de observación y escala; después justifico las transformaciones de representación y verifico su conservación. Mantengo separados plan, objetivos, programas, metas e indicadores: sus cardinalidades y avances no son intercambiables.

Reconozco únicamente los perfiles implementados. Conservo las lecturas originales y las correspondencias de campos para explicar qué homologué. El corte procede de la configuración o del último periodo local común con Excel de ambas fuentes. Una actualización del calendario no demuestra por sí sola compatibilidad de esquema.

**Lectura de los resultados.** La metodología describe el procedimiento para cualquier corte compatible. Los resúmenes que aparecen al ejecutar las celdas describen exclusivamente la fuente seleccionada. Para actualizar el estudio ejecuto el notebook desde el inicio; las salidas guardadas no se actualizan por abrir el archivo. Mantengo las revisiones anteriores en un anexo histórico, sin atribuirlas al corte actual.



In [1]:
# %pip install pandas openpyxl

### Selección de la fuente local

En esta celda resuelvo la carpeta y el corte, identifico un Excel inequívoco y calculo su huella. Si uso la selección automática, exijo un periodo común de las fuentes numéricas. No interpreto una ruta válida como prueba de compatibilidad del contenido.


In [2]:
from pathlib import Path
from datetime import date, datetime, timezone
import hashlib
import json
import platform
import re
import unicodedata
import numpy as np
import pandas as pd
import openpyxl
from IPython.display import display, Markdown

# Buscar la carpeta del proyecto, también al abrir un notebook desde una subcarpeta.
import os
import sys
sys.dont_write_bytecode = True
_carpeta_indicada = os.environ.get("PROYECTO_RAIZ")
_candidatas = ([Path(_carpeta_indicada).expanduser().resolve()] if _carpeta_indicada
               else [Path.cwd().resolve(), *Path.cwd().resolve().parents])
RAIZ = next((p for p in _candidatas if (p / "rutas.py").is_file()
             and (p / "01_descarga_ultimo_corte.ipynb").is_file()), None)
if RAIZ is None:
    raise FileNotFoundError("Abre Jupyter desde la carpeta del proyecto.")
if str(RAIZ) not in sys.path:
    sys.path.insert(0, str(RAIZ))
from rutas import raiz_proyecto, corte_local_comun
RAIZ = raiz_proyecto()
BRONCE = RAIZ / "Capa Bronce"
CORTE = None  # None: último corte local; o fecha ISO, por ejemplo "2026-06-30".
ARCHIVO = None  # None: único Excel del corte; o nombre exacto si hay varios.
AVANCES = ["AVANCE_VIGENCIA", "AVANCE_TRANSCURRIDO", "AVANCE_PLAN"]
SALIDAS = {
    "PlanDesarrollo": "avance_plan.csv",
    "ObjetivoEstratégico": "avance_objetivo_estrategico.csv",
    "Programa": "avance_programa.csv",
    "MetaProducto": "avance_meta_producto.csv",
    "Indicadores": "avance_indicadores.csv",
    "DiccionarioDeDatos": "diccionario_avance_plan.csv",
}
CLAVES = {
    "PlanDesarrollo": ["PLAN_CODIGO", "FECHA_CORTE"],
    "ObjetivoEstratégico": ["PLAN_CODIGO", "PLAN_NIVEL1_CODIGO", "FECHA_CORTE"],
    "Programa": ["PLAN_CODIGO", "PLAN_NIVEL1_CODIGO", "PLAN_NIVEL2_CODIGO", "FECHA_CORTE"],
    "MetaProducto": ["PLAN_CODIGO", "PLAN_META_PRODUCTO_CODIGO", "FECHA_CORTE"],
    "Indicadores": ["PLAN_CODIGO", "PLAN_INDICADOR_CODIGO", "FECHA_CORTE"],
    "DiccionarioDeDatos": ["Columna"],
}
if not BRONCE.is_dir():
    raise FileNotFoundError("Ejecuta desde la raíz del proyecto. Falta Capa Bronce.")
cortes = sorted(p for p in BRONCE.iterdir() if p.is_dir() and re.fullmatch(r"\d{4}-\d{2}-\d{2}", p.name)
                and list((p / "avance_plan").glob("*.xlsx")))
if not cortes:
    raise FileNotFoundError("No hay cortes con Excel estructurado de Avance del Plan.")
carpeta = BRONCE / (CORTE if CORTE else corte_local_comun(RAIZ))
fecha_corte = date.fromisoformat(carpeta.name)
carpeta = carpeta / "avance_plan"
candidatos = sorted(p for p in carpeta.glob("*.xlsx") if not p.name.startswith("~$"))
if ARCHIVO:
    candidatos = [p for p in candidatos if p.name == ARCHIVO]
if len(candidatos) != 1:
    raise ValueError(f"Se requiere un único Excel. Revisa CORTE/ARCHIVO. Candidatos: {candidatos}")
fuente = candidatos[0]
hash_original = hashlib.sha256(fuente.read_bytes()).hexdigest()
DESTINO = RAIZ / "Capa Silver" / fecha_corte.isoformat() / "avance_plan"
print("Fuente:", fuente.relative_to(RAIZ))
print("Fecha de corte:", fecha_corte, "| SHA-256:", hash_original)
print("Entorno:", platform.python_version(), "pandas", pd.__version__, "openpyxl", openpyxl.__version__)

Fuente: Capa Bronce\2026-06-30\avance_plan\01_avancefisico20260630_d6da6ba83a.xlsx
Fecha de corte: 2026-06-30 | SHA-256: c7464dc5bfdbc96d5bd39340c4b0958d38b4394727670cf4bfd54719cae543a0
Entorno: 3.14.7 pandas 3.1.0 openpyxl 3.1.5


## 2. Reconocimiento del esquema y de la escala

Identifico el perfil del Excel y conservo los DataFrames fuente antes de homologar nombres. Determino la escala por el esquema y el formato de las celdas, no por el tamaño del número. Cuando corresponde convierto porcentajes entre cien; cuando ya son fracciones conservo su valor interno.

En un perfil de pivotes separo los totales generales y recupero la jerarquía desde el detalle mediante claves verificadas. Conservo los avances agregados publicados y no los sustituyo por medias simples. Si no existe diccionario o fecha interna, hago explícita esa ausencia y el origen del corte. Ante un perfil o una escala desconocidos detengo la carga.



In [3]:
def adaptar_avance(fuente, fecha_corte):
    """Reconoce únicamente los esquemas observados; conserva la lectura fuente."""
    from pathlib import Path
    from datetime import date, datetime
    import re
    import pandas as pd
    import openpyxl

    libro = openpyxl.load_workbook(fuente, data_only=False)
    originales = pd.read_excel(fuente, sheet_name=None, dtype=object,
                              keep_default_na=False, na_values=[''])
    requeridas = {'PlanDesarrollo', 'ObjetivoEstratégico', 'Programa', 'MetaProducto', 'Indicadores'}
    alias = {'codigo_pd': 'PLAN_CODIGO', 'nombre_pd': 'PLAN_NOMBRE', 'sigla_pd': 'PLAN_SIGLA',
             'codigo_componente_n1': 'PLAN_NIVEL1_CODIGO', 'nombre_componente_n1': 'PLAN_NIVEL1_NOMBRE',
             'codigo_componente_n2': 'PLAN_NIVEL2_CODIGO', 'nombre_componente_n2': 'PLAN_NIVEL2_NOMBRE',
             'codigo_mpd': 'PLAN_META_PRODUCTO_CODIGO', 'nombre_mpd': 'PLAN_META_PRODUCTO_NOMBRE',
             'codigo_meta_producto': 'PLAN_META_PRODUCTO_CODIGO', 'nombre_meta_producto': 'PLAN_META_PRODUCTO_NOMBRE',
             'codigo_indicador': 'PLAN_INDICADOR_CODIGO', 'nombre_indicador': 'PLAN_INDICADOR_NOMBRE',
             'vigencia': 'VIGENCIA_CORTE', 'vigencia_corte': 'VIGENCIA_CORTE',
             'vigencia_inicio': 'VIGENCIA_INICIO', 'vigencia_fin': 'VIGENCIA_FIN',
             'fechacorte': 'FECHA_CORTE', 'fechacorte_fisico': 'FECHA_CORTE', 'fecha_corte': 'FECHA_CORTE',
             'avancevigencia': 'AVANCE_VIGENCIA', 'avancetranscurrido': 'AVANCE_TRANSCURRIDO',
             'avanceplan': 'AVANCE_PLAN', 'avance_vigencia': 'AVANCE_VIGENCIA',
             'avance_transcurrido': 'AVANCE_TRANSCURRIDO', 'avance_plan': 'AVANCE_PLAN',
             'ods': 'ODS', 'ods_meta': 'ODS_META'}
    escalas, correspondencias, auxiliares = {}, [], {}
    campos_avance = ['AVANCE_VIGENCIA', 'AVANCE_TRANSCURRIDO', 'AVANCE_PLAN']
    if requeridas.issubset(originales):
        brutas = {}
        historico = 'codigo_pd' in originales['PlanDesarrollo']
        version = 'tabular_historico_0_100' if historico else 'tabular_fracciones'
        if not historico and 'PLAN_CODIGO' not in originales['PlanDesarrollo']:
            raise ValueError('Esquema tabular no reconocido; revisar encabezados.')
        for hoja, original in originales.items():
            if hoja not in requeridas | {'DiccionarioDeDatos'}:
                auxiliares[hoja] = original
                continue
            df = original.rename(columns=alias if historico else {}).copy()
            if not df.columns.is_unique:
                raise ValueError(f'La homologación genera campos duplicados: {hoja}')
            for origen, destino in zip(original.columns, df.columns):
                correspondencias.append({'hoja': hoja, 'campo_fuente': origen, 'campo_preparado': destino})
            if 'FECHA_CORTE' in df:
                # DD/MM/AAAA (estado) es un formato explícito observado, no una inferencia regional.
                def fecha_explicita(v):
                    if isinstance(v, (date, datetime, pd.Timestamp)):
                        return pd.Timestamp(v)
                    m = re.fullmatch(r'(\d{2}/\d{2}/\d{4})(?:\s*\([^()]*\))?', str(v).strip())
                    if not m:
                        raise ValueError(f'Fecha histórica no reconocida: {v!r}')
                    return pd.to_datetime(m[1], format='%d/%m/%Y', errors='raise')
                df['FECHA_CORTE'] = df['FECHA_CORTE'].map(fecha_explicita)
            escalas[hoja] = {}
            for campo in campos_avance:
                if campo not in df:
                    continue
                columna = list(df.columns).index(campo) + 1
                formatos = {libro[hoja].cell(i + 2, columna).number_format for i in df.index}
                if historico:
                    escalas[hoja][campo] = 'porcentaje_0_100'
                elif all('%' in f for f in formatos):
                    escalas[hoja][campo] = 'fraccion'
                elif fecha_corte.isoformat() in {'2025-03-31','2025-06-30'} and formatos.issubset({'#,##0.00','0.00','General'}):
                    escalas[hoja][campo] = 'porcentaje_0_100'
                    version = 'tabular_2025_0_100'
                else:
                    raise ValueError(f'Escala no documentada: {hoja}/{campo}: {formatos}')
            brutas[hoja] = df
    elif {'PlanDesarrollo', 'Objetivo', 'Programa', 'Meta', 'Indicador', 'datos'}.issubset(originales):
        version = 'pivotes_y_catalogo_2026'
        detalle = originales['datos']
        if 'PLAN_CODIGO' not in detalle:
            raise ValueError('Falta el plan en el detalle del archivo de pivotes.')
        auxiliares['pie_detalle_sin_clave'] = detalle.loc[detalle.PLAN_CODIGO.isna()].copy()
        detalle = detalle.loc[detalle.PLAN_CODIGO.notna()].copy()
        if detalle.empty or detalle.PLAN_CODIGO.nunique() != 1:
            raise ValueError('El catálogo de pivotes no identifica un único plan.')
        auxiliares['detalle_entidad_indicador'] = detalle
        identidad = {}
        for col in ['PLAN_CODIGO', 'PLAN_NOMBRE', 'PLAN_SIGLA']:
            if detalle[col].nunique() != 1:
                raise ValueError(f'Identidad de plan ambigua: {col}')
            identidad[col] = detalle[col].iloc[0]
        nombres = {'PlanDesarrollo':'PlanDesarrollo', 'Objetivo':'ObjetivoEstratégico',
                   'Programa':'Programa', 'Meta':'MetaProducto', 'Indicador':'Indicadores'}
        etiquetas = {'Suma de Vigencia':'AVANCE_VIGENCIA', 'Suma de Transcurrido':'AVANCE_TRANSCURRIDO',
                     'Suma de Plan':'AVANCE_PLAN'}
        niveles = ['PLAN_NIVEL1_CODIGO', 'PLAN_NIVEL1_NOMBRE', 'PLAN_NIVEL2_CODIGO',
                   'PLAN_NIVEL2_NOMBRE', 'PLAN_META_PRODUCTO_CODIGO', 'PLAN_META_PRODUCTO_NOMBRE',
                   'PLAN_INDICADOR_CODIGO', 'PLAN_INDICADOR_NOMBRE']
        brutas = {}
        for fuente_hoja, salida_hoja in nombres.items():
            df = originales[fuente_hoja].copy()
            if not set(etiquetas).issubset(df):
                raise ValueError(f'Faltan avances de pivote: {fuente_hoja}')
            codigo = next((c for c in df if c.endswith('_CODIGO')), None)
            if codigo:
                totales = df[codigo].astype(str).str.casefold().eq('total general')
                if int(totales.sum()) != 1:
                    raise ValueError(f'Total general inesperado: {fuente_hoja}')
                auxiliares[f'total_{fuente_hoja}'] = df.loc[totales].copy()
                df = df.loc[~totales].copy()
                if df[codigo].isna().any() or df[codigo].duplicated().any():
                    raise ValueError(f'Claves de pivote no únicas: {fuente_hoja}')
                # Recupero únicamente la jerarquía; no agrego ni recalculo avances oficiales.
                posicion = niveles.index(codigo)
                padres = niveles[:posicion + 2]
                catalogo = detalle[padres].drop_duplicates()
                if catalogo[codigo].duplicated().any():
                    raise ValueError(f'Jerarquía ambigua en el catálogo: {codigo}')
                complementarias = [c for c in padres if c not in df]
                df = df.merge(catalogo[[codigo, *complementarias]], on=codigo,
                              how='left', validate='one_to_one', indicator=True)
                if not df['_merge'].eq('both').all():
                    raise ValueError(f'Pivote sin correspondencia en detalle: {fuente_hoja}')
                df = df.drop(columns='_merge')
            df = df.rename(columns=etiquetas)
            for col, valor in identidad.items():
                df[col] = valor
            df['VIGENCIA_CORTE'] = fecha_corte.year
            df['FECHA_CORTE'] = pd.Timestamp(fecha_corte)
            # El corte proviene de la publicación/carpeta: esta versión no lo incluye como campo.
            escalas[salida_hoja] = {}
            for col in campos_avance:
                origen = next(k for k,v in etiquetas.items() if v == col)
                j = list(originales[fuente_hoja].columns).index(origen) + 1
                formatos = {libro[fuente_hoja].cell(i + 2, j).number_format for i in originales[fuente_hoja].index}
                if not all('%' in f for f in formatos):
                    raise ValueError(f'Formato porcentual de pivote no confirmado: {fuente_hoja}/{origen}')
                escalas[salida_hoja][col] = 'fraccion'
            brutas[salida_hoja] = df.reset_index(drop=True)
        correspondencias.append({'hoja':'pivotes','campo_fuente':'Suma de ...',
                                  'campo_preparado':'AVANCE_*; valores oficiales sin recalcular'})
    else:
        libro.close()
        raise ValueError('Esquema de Avance del Plan no compatible. No equiparo metas sectoriales del plan anterior con metas producto.')
    if brutas['PlanDesarrollo']['PLAN_CODIGO'].dropna().astype(str).str.replace(r'\.0$', '', regex=True).nunique() != 1:
        libro.close()
        raise ValueError('La fuente no identifica un único plan; no mezclo planes en una misma tabla.')
    inventario = []
    for hoja in libro:
        formulas = sum(c.data_type == 'f' for fila in hoja for c in fila)
        errores = sum(c.data_type == 'e' for fila in hoja for c in fila)
        inventario.append({'hoja':hoja.title,'filas_sin_encabezado':hoja.max_row-1,
                           'columnas':hoja.max_column,'formulas':formulas,'errores_excel':errores,
                           'combinaciones':len(hoja.merged_cells.ranges)})
        usada = hoja.title in requeridas or (version=='pivotes_y_catalogo_2026' and hoja.title in nombres)
        if usada and (formulas or errores):
            libro.close()
            raise ValueError(f'Fórmulas o errores en la tabla consumida: {hoja.title}')
    return libro, originales, brutas, escalas, pd.DataFrame(correspondencias), auxiliares, version, pd.DataFrame(inventario)

libro, brutas_fuente, brutas, escalas_avance, correspondencias, auxiliares_fuente, version_esquema, inventario = adaptar_avance(fuente, fecha_corte)
print("Esquema reconocido:", version_esquema)
display(inventario)
display(correspondencias)
display(pd.DataFrame([{"hoja": h, "campo": col, "escala_fuente": escala}
                      for h, cols in escalas_avance.items() for col, escala in cols.items()]))
for nombre, df in brutas.items():
    display(Markdown(f"**{nombre}:** {len(df)} filas × {len(df.columns)} columnas"))
    display(df.head(2))
if "DiccionarioDeDatos" not in brutas:
    print("La fuente no incluye diccionario: no fabrico una hoja oficial para este corte.")


Esquema reconocido: tabular_fracciones


,hoja,filas_sin_encabezado,columnas,formulas,errores_excel,combinaciones
0,PlanDesarrollo,1,10,0,0,0
1,ObjetivoEstratégico,5,10,0,0,0
2,Programa,39,12,0,0,0
3,MetaProducto,425,16,0,0,0
4,Indicadores,443,16,0,0,0
5,DiccionarioDeDatos,16,4,0,0,0


,hoja,campo_fuente,campo_preparado
0,PlanDesarrollo,PLAN_CODIGO,PLAN_CODIGO
1,PlanDesarrollo,PLAN_NOMBRE,PLAN_NOMBRE
2,PlanDesarrollo,PLAN_SIGLA,PLAN_SIGLA
3,PlanDesarrollo,VIGENCIA_INICIO,VIGENCIA_INICIO
4,PlanDesarrollo,VIGENCIA_FIN,VIGENCIA_FIN
...,...,...,...
63,Indicadores,AVANCE_PLAN,AVANCE_PLAN
64,DiccionarioDeDatos,Columna,Columna
65,DiccionarioDeDatos,Tipo,Tipo
66,DiccionarioDeDatos,Descripción,Descripción


,hoja,campo,escala_fuente
0,PlanDesarrollo,AVANCE_VIGENCIA,fraccion
1,PlanDesarrollo,AVANCE_TRANSCURRIDO,fraccion
2,PlanDesarrollo,AVANCE_PLAN,fraccion
3,ObjetivoEstratégico,AVANCE_VIGENCIA,fraccion
4,ObjetivoEstratégico,AVANCE_TRANSCURRIDO,fraccion
5,ObjetivoEstratégico,AVANCE_PLAN,fraccion
6,Programa,AVANCE_VIGENCIA,fraccion
7,Programa,AVANCE_TRANSCURRIDO,fraccion
8,Programa,AVANCE_PLAN,fraccion
9,MetaProducto,AVANCE_VIGENCIA,fraccion


**PlanDesarrollo:** 1 filas × 10 columnas

,PLAN_CODIGO,PLAN_NOMBRE,PLAN_SIGLA,VIGENCIA_INICIO,VIGENCIA_FIN,VIGENCIA_CORTE,FECHA_CORTE,AVANCE_VIGENCIA,AVANCE_TRANSCURRIDO,AVANCE_PLAN
0,17,Bogotá Camina Segura,BCS,2024,2027,2026,2026-06-30,0.512211,0.710886,0.533338


... (6 outputs left out)

**Indicadores:** 443 filas × 16 columnas

,PLAN_CODIGO,PLAN_NOMBRE,PLAN_SIGLA,PLAN_NIVEL1_CODIGO,PLAN_NIVEL1_NOMBRE,PLAN_NIVEL2_CODIGO,PLAN_NIVEL2_NOMBRE,PLAN_META_PRODUCTO_CODIGO,PLAN_META_PRODUCTO_NOMBRE,PLAN_INDICADOR_CODIGO,PLAN_INDICADOR_NOMBRE,VIGENCIA_CORTE,FECHA_CORTE,AVANCE_VIGENCIA,AVANCE_TRANSCURRIDO,AVANCE_PLAN
0,17,Bogotá Camina Segura,BCS,1,Bogotá avanza en su seguridad,1,Diálogo social y cultura ciudadana para la con...,1929,Desarrollar 1200 Intervención(es) del Sistema ...,3851,Número de intervenciones pedagógicas y de difu...,2026,2026-06-30,0.15,0.716667,0.5375
1,17,Bogotá Camina Segura,BCS,1,Bogotá avanza en su seguridad,1,Diálogo social y cultura ciudadana para la con...,1930,Fortalecer 1 Programa(s) de atención integral ...,3852,Programa de atención integral en el marco del ...,2026,2026-06-30,0.24,0.683333,0.41


**DiccionarioDeDatos:** 16 filas × 4 columnas

,Columna,Tipo,Descripción,Observación
0,PLAN_CODIGO,Numérico,Código del Plan de Desarrollo,Codificación acorde con BogData de Secretaría ...
1,PLAN_NOMBRE,Texto,Nombre del Plan de Desarrollo,NaN


### Inventario del archivo seleccionado

Antes de interpretar los avances describo las tablas presentes y su tamaño. El inventario siguiente se calcula desde la lectura de esta ejecución; no presupongo un número constante de objetivos, programas, metas o indicadores ni la presencia de un diccionario.



### Lectura calculada del inventario

Calculo esta lectura con las variables de la ejecución, sin fijar conteos ni conclusiones de otro corte.


In [13]:
_inventario_prosa = "; ".join(f"{_h}: {len(_df)} filas y {len(_df.columns)} campos" for _h, _df in brutas.items())
_diccionario_prosa = "incluye diccionario" if "DiccionarioDeDatos" in brutas else "no incluye diccionario oficial"
display(Markdown(f"**Corte seleccionado: {fecha_corte.isoformat()}.** Identifico {_inventario_prosa}. La fuente {_diccionario_prosa}. Examino cada nivel por separado y no uso un conteo anterior como condición de aceptación."))


**Corte seleccionado: 2026-06-30.** Identifico PlanDesarrollo: 1 filas y 10 campos; ObjetivoEstratégico: 5 filas y 10 campos; Programa: 39 filas y 12 campos; MetaProducto: 425 filas y 16 campos; Indicadores: 443 filas y 16 campos; DiccionarioDeDatos: 16 filas y 4 campos. La fuente incluye diccionario. Examino cada nivel por separado y no uso un conteo anterior como condición de aceptación.

## 3. Exploración de calidad antes de transformar

Reviso vacíos, tipos, valores distintos, duplicados y claves. Cuento los textos que cambiarían al normalizar espacios y Unicode y examino distribución, ceros y extremos de los avances por nivel.

Para el diagnóstico expreso las vistas de avance en proporciones según la escala reconocida. Mantengo intactas las tablas de entrada. La conversión posterior se realiza y comprueba sobre una copia.



In [4]:
def espacio_limpio(valor):
    if isinstance(valor, str):
        return re.sub(r"\s+", " ", unicodedata.normalize("NFC", valor).replace("\u00a0", " ")).strip()
    return valor

def perfil(tablas):
    filas = []
    resumen = []
    for hoja, df in tablas.items():
        resumen.append({"hoja": hoja, "filas": len(df), "columnas": len(df.columns),
                        "filas_vacias": int(df.isna().all(axis=1).sum()),
                        "duplicados_exactos": int(df.duplicated().sum()),
                        "claves_duplicadas": int(df.duplicated(CLAVES[hoja]).sum()),
                        "claves_nulas": int(df[CLAVES[hoja]].isna().any(axis=1).sum())})
        for col in df:
            s = df[col]
            filas.append({"hoja": hoja, "columna": col, "tipo_pandas": str(s.dtype),
                          "tipos_celda": ", ".join(sorted({type(v).__name__ for v in s.dropna()})),
                          "nulos": int(s.isna().sum()), "fraccion_nulos": s.isna().mean(),
                          "distintos": s.nunique(dropna=True),
                          "textos_a_normalizar": sum(isinstance(v, str) and espacio_limpio(v) != v for v in s),
                          "textos_vacios": sum(isinstance(v, str) and not espacio_limpio(v) for v in s)})
    return pd.DataFrame(resumen), pd.DataFrame(filas)

resumen_antes, perfil_antes = perfil(brutas)
display(resumen_antes)
display(perfil_antes.query("nulos > 0 or textos_a_normalizar > 0 or textos_vacios > 0"))
estadisticas = []
for hoja, df in brutas.items():
    if all(c in df for c in AVANCES):
        numericas = df[AVANCES].apply(pd.to_numeric, errors="raise")
        for campo in AVANCES:
            if escalas_avance[hoja][campo] == "porcentaje_0_100":
                numericas[campo] = numericas[campo] / 100
        tabla = numericas.describe(percentiles=[.25, .5, .75]).T.reset_index(names="variable")
        tabla.insert(0, "hoja", hoja)
        tabla["ceros"] = [int((numericas[c] == 0).sum()) for c in tabla.variable]
        tabla["fuera_0_1"] = [int(((numericas[c] < 0) | (numericas[c] > 1)).sum()) for c in tabla.variable]
        estadisticas.append(tabla)
estadisticas = pd.concat(estadisticas, ignore_index=True)
display(estadisticas[["hoja", "variable", "count", "min", "25%", "50%", "75%", "max", "ceros", "fuera_0_1"]])


,hoja,filas,columnas,filas_vacias,duplicados_exactos,claves_duplicadas,claves_nulas
0,PlanDesarrollo,1,10,0,0,0,0
1,ObjetivoEstratégico,5,10,0,0,0,0
2,Programa,39,12,0,0,0,0
3,MetaProducto,425,16,0,0,0,0
4,Indicadores,443,16,0,0,0,0
5,DiccionarioDeDatos,16,4,0,0,0,0


,hoja,columna,tipo_pandas,tipos_celda,nulos,fraccion_nulos,distintos,textos_a_normalizar,textos_vacios
40,MetaProducto,PLAN_META_PRODUCTO_NOMBRE,object,str,0,0.0000,425,2,0
46,MetaProducto,ODS,object,str,0,0.0000,18,18,0
47,MetaProducto,ODS_META,object,str,0,0.0000,80,2,0
56,Indicadores,PLAN_META_PRODUCTO_NOMBRE,object,str,0,0.0000,425,2,0
58,Indicadores,PLAN_INDICADOR_NOMBRE,object,str,0,0.0000,439,1,0
67,DiccionarioDeDatos,Observación,object,str,15,0.9375,1,0,0


,hoja,variable,count,min,25%,50%,75%,max,ceros,fuera_0_1
0,PlanDesarrollo,AVANCE_VIGENCIA,1.0,0.512211,0.512211,0.512211,0.512211,0.512211,0,0
1,PlanDesarrollo,AVANCE_TRANSCURRIDO,1.0,0.710886,0.710886,0.710886,0.710886,0.710886,0,0
2,PlanDesarrollo,AVANCE_PLAN,1.0,0.533338,0.533338,0.533338,0.533338,0.533338,0,0
3,ObjetivoEstratégico,AVANCE_VIGENCIA,5.0,0.377059,0.417431,0.550452,0.586124,0.638185,0,0
4,ObjetivoEstratégico,AVANCE_TRANSCURRIDO,5.0,0.585931,0.743095,0.748027,0.774000,0.808021,0,0
5,ObjetivoEstratégico,AVANCE_PLAN,5.0,0.433253,0.547746,0.566374,0.577695,0.627550,0,0
6,Programa,AVANCE_VIGENCIA,39.0,0.021739,0.408843,0.496849,0.589958,0.869491,0,0
7,Programa,AVANCE_TRANSCURRIDO,39.0,0.521751,0.697964,0.750000,0.806221,0.970348,0,0
8,Programa,AVANCE_PLAN,39.0,0.361883,0.507781,0.562597,0.610916,0.747789,0,0
9,MetaProducto,AVANCE_VIGENCIA,425.0,0.000000,0.307054,0.500000,0.749415,1.000000,59,0


### Cómo relaciono la exploración con la limpieza

Interpreto los conteos del corte seleccionado antes de aplicar reglas. Si aparecen vacíos, duplicados o valores extremos los hago visibles; no presupongo que deban desaparecer. La ausencia de esos casos en otra publicación no es una regla para esta fuente.

Cuando encuentro variantes de espacios o Unicode normalizo su representación. Los códigos de objetivo y programa se representan como texto de dos dígitos, según el perfil implementado. Un cero de avance se conserva como valor reportado; no lo equiparo con una medición ausente.



### Evidencia calculada para decidir las transformaciones

Calculo esta lectura con las variables de la ejecución, sin fijar conteos ni conclusiones de otro corte.


In [14]:
_tablas_datos_prosa = [h for h in brutas if h != "DiccionarioDeDatos"]
_nulos_prosa = sum(int(brutas[h].isna().sum().sum()) for h in _tablas_datos_prosa)
_duplicados_prosa = int(resumen_antes.loc[resumen_antes.hoja.isin(_tablas_datos_prosa), "duplicados_exactos"].sum())
_textos_prosa = int(perfil_antes.textos_a_normalizar.sum())
_extremos_prosa = int(estadisticas.fuera_0_1.sum())
display(Markdown(f"Para **{fecha_corte.isoformat()}** encuentro **{_nulos_prosa} celdas nulas** en tablas de datos, **{_duplicados_prosa} filas duplicadas exactas** y **{_textos_prosa} celdas de texto candidatas a normalización** en las tablas leídas. Observo **{_extremos_prosa} valores de avance fuera de [0, 1]** tras expresar el diagnóstico en proporciones. Estos conteos justifican examinar sus campos; no autorizan imputar, borrar o recortar valores."))


Para **2026-06-30** encuentro **0 celdas nulas** en tablas de datos, **0 filas duplicadas exactas** y **25 celdas de texto candidatas a normalización** en las tablas leídas. Observo **0 valores de avance fuera de [0, 1]** tras expresar el diagnóstico en proporciones. Estos conteos justifican examinar sus campos; no autorizan imputar, borrar o recortar valores.

## 4. Reglas de preparación y fundamento

| Condición examinada | Decisión | Límite |
|---|---|---|
| Códigos de objetivo y programa | Texto de dos dígitos | No impongo la misma longitud a todos los códigos |
| Espacios y Unicode no uniformes | Normalización de representación | No reescribo significado ni redacción |
| Porcentaje en escala reconocida 0–100 | División entre cien | No deduzco escala desde el máximo |
| Avance ya fraccionario | Conservación del valor interno | Evito una segunda división |
| Fecha explícita | Conversión y contraste con corte y vigencia | No adivino formatos ambiguos |
| Etiquetas ODS presentes | Derivación de sus códigos | No invento etiquetas ausentes |
| Totales generales de pivotes | Separación en auxiliares | No los cuento como observaciones adicionales |
| Vacíos, variantes y extremos | Diagnóstico y conservación | No imputo ni recorto automáticamente |

Aplico estas decisiones a una copia. Las tablas de lectura fuente permanecen disponibles para contrastar cada cambio.



In [5]:
def codigo_texto(v, ancho=None):
    if pd.isna(v):
        return pd.NA
    s = str(v).strip()
    if not re.fullmatch(r"\d+(?:\.0+)?", s):
        raise ValueError(f"Código inesperado: {v!r}")
    s = str(int(s.split(".")[0]))
    if ancho and len(s) > ancho:
        raise ValueError(f"Código excede {ancho} dígitos: {v!r}")
    return s.zfill(ancho) if ancho else s

limpias = {}
cambios = []
for hoja, original in brutas.items():
    df = original.copy(deep=True)
    for col in df:
        nuevos = df[col].map(espacio_limpio)
        n = sum(isinstance(a, str) and a != b for a, b in zip(df[col], nuevos))
        if n:
            cambios.append({"hoja": hoja, "columna": col, "regla": "normalizar_espacios_unicode", "celdas": n})
        df[col] = nuevos.map(lambda v: pd.NA if isinstance(v, str) and v == "" else v)
        if col.endswith("_CODIGO"):
            ancho = 2 if col in ["PLAN_NIVEL1_CODIGO", "PLAN_NIVEL2_CODIGO"] else None
            df[col] = df[col].map(lambda v: codigo_texto(v, ancho)).astype("string")
        elif col.startswith("VIGENCIA_"):
            df[col] = pd.to_numeric(df[col], errors="raise").astype("Int64")
        elif col == "FECHA_CORTE":
            # No se adivinan seriales Excel ni fechas textuales de orden ambiguo.
            if any(not isinstance(v, (datetime, date, pd.Timestamp)) for v in df[col].dropna()):
                raise ValueError(f"Revisar el tipo de fecha en {hoja}")
            df[col] = pd.to_datetime(df[col], errors="raise")
        elif col in AVANCES:
            valores = []
            columna_excel = list(original.columns).index(col) + 1
            for fila, v in enumerate(df[col], start=2):
                escala = escalas_avance[hoja][col]
                if pd.isna(v):
                    valores.append(np.nan)
                elif isinstance(v, str) and v.endswith("%"):
                    valores.append(float(v[:-1].strip().replace(",", ".")) / 100)
                elif isinstance(v, (int, float, np.number)) and not isinstance(v, bool) and escala == "fraccion":
                    valores.append(float(v))
                elif isinstance(v, (int, float, np.number)) and not isinstance(v, bool) and escala == "porcentaje_0_100":
                    valores.append(float(v) / 100)
                else:
                    raise ValueError(f"Escala ambigua en {hoja}!{fila}, {col}: {v!r}, escala {escala}")
            df[col] = pd.Series(valores, index=df.index, dtype="float64")
            if not np.isfinite(df[col].dropna()).all():
                raise ValueError(f"Avances no finitos en {hoja}/{col}")
        else:
            df[col] = df[col].astype("string")
    if hoja == "MetaProducto" and {"ODS", "ODS_META"}.issubset(df):
        df["ODS_CODIGO"] = df["ODS"].str.extract(r"^(\d{2})[.]", expand=False).astype("string")
        df["ODS_META_CODIGO"] = df["ODS_META"].str.extract(r"^(\d{2}[.](?:\d{2}|[A-Za-z]))[.]", expand=False).astype("string")
        if df[["ODS_CODIGO", "ODS_META_CODIGO"]].isna().any().any():
            raise ValueError("Etiquetas ODS con estructura inesperada; revisar antes de derivar códigos.")
        if not df["ODS_CODIGO"].equals(df["ODS_META_CODIGO"].str[:2].astype("string")):
            raise ValueError("El ODS y su meta no son coherentes.")
        if not pd.to_numeric(df["ODS_CODIGO"]).between(1, 17).all():
            raise ValueError("Código ODS fuera de 01–17.")
    limpias[hoja] = df
display(pd.DataFrame(cambios))
display(limpias["MetaProducto"].head(3))

,hoja,columna,regla,celdas
0,MetaProducto,PLAN_META_PRODUCTO_NOMBRE,normalizar_espacios_unicode,2
1,MetaProducto,ODS,normalizar_espacios_unicode,18
2,MetaProducto,ODS_META,normalizar_espacios_unicode,2
3,Indicadores,PLAN_META_PRODUCTO_NOMBRE,normalizar_espacios_unicode,2
4,Indicadores,PLAN_INDICADOR_NOMBRE,normalizar_espacios_unicode,1


,PLAN_CODIGO,PLAN_NOMBRE,PLAN_SIGLA,PLAN_NIVEL1_CODIGO,PLAN_NIVEL1_NOMBRE,PLAN_NIVEL2_CODIGO,PLAN_NIVEL2_NOMBRE,PLAN_META_PRODUCTO_CODIGO,PLAN_META_PRODUCTO_NOMBRE,VIGENCIA_CORTE,FECHA_CORTE,AVANCE_VIGENCIA,AVANCE_TRANSCURRIDO,AVANCE_PLAN,ODS,ODS_META,ODS_CODIGO,ODS_META_CODIGO
0,17,Bogotá Camina Segura,BCS,01,Bogotá avanza en su seguridad,01,Diálogo social y cultura ciudadana para la con...,1929,Desarrollar 1200 Intervención(es) del Sistema ...,2026,2026-06-30,0.433333,0.811111,0.608333,"16. Paz, Justicia e Instituciones Sólidas",16.03. Promover el estado de derecho en los pl...,16,16.03
1,17,Bogotá Camina Segura,BCS,01,Bogotá avanza en su seguridad,01,Diálogo social y cultura ciudadana para la con...,1930,Fortalecer 1 Programa(s) de atención integral ...,2026,2026-06-30,0.520000,0.800000,0.480000,"16. Paz, Justicia e Instituciones Sólidas",16.07. Garantizar la adopción en todos los niv...,16,16.07
2,17,Bogotá Camina Segura,BCS,01,Bogotá avanza en su seguridad,01,Diálogo social y cultura ciudadana para la con...,1931,Fortalecer 1 Programa(s) junto con sus estrate...,2026,2026-06-30,0.320000,0.716667,0.430000,"16. Paz, Justicia e Instituciones Sólidas",16.01. Reducir significativamente todas las fo...,16,16.01


## 5. Comprobación del efecto de las decisiones

Repito el perfil y verifico conservación de observaciones, claves, fechas y relaciones entre niveles. Contrasto avances con su entrada, admitiendo solo la conversión documentada. Si la fuente aporta inicio y fin del plan, compruebo sus límites; si no los aporta, no los imputo.

Los cruces deben conservar filas y referencias. Registro las variantes de etiqueta sin elegir automáticamente una redacción oficial ni exigir una relación uno a uno entre metas e indicadores.



In [15]:
resumen_despues, perfil_despues = perfil(limpias)
display(resumen_despues)
for hoja, df in limpias.items():
    assert len(df) == len(brutas[hoja]), f"Pérdida de filas: {hoja}"
    assert set(brutas[hoja].columns).issubset(df.columns)
    if df[CLAVES[hoja]].isna().any().any() or df.duplicated(CLAVES[hoja]).any():
        raise ValueError(f"Claves nulas o duplicadas: {hoja}; revisar sin eliminar automáticamente.")
    if "FECHA_CORTE" in df:
        assert df["FECHA_CORTE"].notna().all()
        assert (df["FECHA_CORTE"] == pd.Timestamp(fecha_corte)).all(), f"Corte incoherente: {hoja}"
        assert (df["VIGENCIA_CORTE"] == fecha_corte.year).all()
        for col in AVANCES:
            original = brutas[hoja][col]
            # El archivo inspeccionado es numérico: control independiente de no dividir dos veces.
            if all(isinstance(v, (int, float)) for v in original.dropna()):
                factor = 100 if escalas_avance[hoja][col] == "porcentaje_0_100" else 1
                np.testing.assert_allclose(df[col], pd.to_numeric(original) / factor, rtol=0, atol=0, equal_nan=True)
            assert df[col].isna().sum() == original.isna().sum()
plan = limpias["PlanDesarrollo"]
if {"VIGENCIA_INICIO", "VIGENCIA_FIN"}.issubset(plan):
    assert ((plan.VIGENCIA_INICIO <= plan.VIGENCIA_CORTE) & (plan.VIGENCIA_CORTE <= plan.VIGENCIA_FIN)).all()
else:
    print("La versión fuente no aporta inicio y fin del plan; no los imputo.")

relaciones = [
    ("ObjetivoEstratégico", "PlanDesarrollo", CLAVES["PlanDesarrollo"]),
    ("Programa", "ObjetivoEstratégico", CLAVES["ObjetivoEstratégico"]),
    ("MetaProducto", "Programa", CLAVES["Programa"]),
    ("Indicadores", "MetaProducto", ["PLAN_CODIGO", "PLAN_NIVEL1_CODIGO", "PLAN_NIVEL2_CODIGO", "PLAN_META_PRODUCTO_CODIGO", "FECHA_CORTE"]),
]
control_relaciones = []
for hija, padre, columnas in relaciones:
    union = limpias[hija].merge(limpias[padre][columnas], on=columnas, how="left", validate="many_to_one", indicator=True)
    huerfanos = int((union["_merge"] == "left_only").sum())
    control_relaciones.append({"tabla": hija, "referencia": padre, "registros_sin_padre": huerfanos})
    if huerfanos:
        raise ValueError(f"Hay registros sin referencia: {hija} -> {padre}")
display(pd.DataFrame(control_relaciones))

incidencias = []
for campo in ["PLAN_NOMBRE", "PLAN_NIVEL1_NOMBRE", "PLAN_NIVEL2_NOMBRE", "PLAN_META_PRODUCTO_NOMBRE", "PLAN_INDICADOR_NOMBRE", "ODS", "ODS_META"]:
    cod = campo.replace("NOMBRE", "CODIGO") if campo.endswith("NOMBRE") else campo + "_CODIGO"
    partes = [df[[cod, campo]].assign(hoja=hoja) for hoja, df in limpias.items() if cod in df and campo in df]
    if not partes:
        continue
    etiquetas = pd.concat(partes, ignore_index=True).drop_duplicates()
    conflictos = etiquetas.groupby(cod)[campo].nunique()
    for _, r in etiquetas[etiquetas[cod].isin(conflictos[conflictos > 1].index)].iterrows():
        incidencias.append({"tipo": "etiquetas_distintas", "hoja": r["hoja"], "campo": campo, "codigo_o_fila": r[cod], "valor": r[campo]})
for hoja, df in limpias.items():
    for col in AVANCES:
        if col in df:
            for i, v in df.loc[(df[col] < 0) | (df[col] > 1), col].items():
                incidencias.append({"tipo": "avance_fuera_0_1", "hoja": hoja, "campo": col, "codigo_o_fila": str(i + 2), "valor": v})
incidencias = pd.DataFrame(incidencias, columns=["tipo", "hoja", "campo", "codigo_o_fila", "valor"])
display(incidencias)
display(limpias["Indicadores"].groupby("PLAN_META_PRODUCTO_CODIGO").size().value_counts().rename_axis("indicadores_por_meta").rename("numero_metas").to_frame())
assert hashlib.sha256(fuente.read_bytes()).hexdigest() == hash_original, "El original fue modificado"
print("Validaciones completadas. Las incidencias conservan el dato original.")

# Conclusiones calculadas a partir del archivo, sin asumir resultados futuros.
hojas_datos = [h for h in limpias if h != "DiccionarioDeDatos"]
nulos_datos = sum(int(brutas[h].isna().sum().sum()) for h in hojas_datos)
duplicados = int(resumen_antes.loc[resumen_antes.hoja.isin(hojas_datos), "duplicados_exactos"].sum())
celdas_texto = sum(c["celdas"] for c in cambios)
fuera_rango = int(estadisticas["fuera_0_1"].sum())
display(Markdown(f"""### Resultados del diagnóstico y de la limpieza
- Corte analizado: **{fecha_corte}**; {len(hojas_datos)} tablas de datos; diccionario oficial {"presente" if "DiccionarioDeDatos" in limpias else "ausente"}.
- Celdas nulas en las tablas de datos antes de limpiar: **{nulos_datos}**; filas duplicadas exactas: **{duplicados}**.
- Celdas de texto normalizadas: **{celdas_texto}**. Los códigos se tipificaron de forma consistente.
- Valores de avance fuera de [0, 1]: **{fuera_rango}**. No se recortaron ni imputaron valores.
- Se conservaron todas las filas y se verificaron claves, fechas y relaciones jerárquicas.
- El detalle de las variantes de etiquetas está en la tabla de incidencias anterior; su redacción se conserva.

Estas comprobaciones describen integridad técnica del archivo; no certifican la exactitud sustantiva de lo reportado por las entidades ni sustituyen las fichas metodológicas de los indicadores.
"""))


Validaciones completadas. Las incidencias conservan el dato original.


,hoja,filas,columnas,filas_vacias,duplicados_exactos,claves_duplicadas,claves_nulas
0,PlanDesarrollo,1,10,0,0,0,0
1,ObjetivoEstratégico,5,10,0,0,0,0
2,Programa,39,12,0,0,0,0
3,MetaProducto,425,18,0,0,0,0
4,Indicadores,443,16,0,0,0,0
5,DiccionarioDeDatos,16,4,0,0,0,0


,tabla,referencia,registros_sin_padre
0,ObjetivoEstratégico,PlanDesarrollo,0
1,Programa,ObjetivoEstratégico,0
2,MetaProducto,Programa,0
3,Indicadores,MetaProducto,0


,tipo,hoja,campo,codigo_o_fila,valor
0,etiquetas_distintas,Programa,PLAN_NIVEL2_NOMBRE,01,Diálogo social y cultura ciudadana para la con...
1,etiquetas_distintas,MetaProducto,PLAN_NIVEL2_NOMBRE,01,Diálogo social y cultura ciudadana para la con...
2,etiquetas_distintas,Indicadores,PLAN_NIVEL2_NOMBRE,01,Diálogo social y cultura ciudadana para la con...
3,etiquetas_distintas,MetaProducto,ODS,04,04. Educación de Calidad
4,etiquetas_distintas,MetaProducto,ODS,11,11. Ciudades y Comunidades Sostenibles
5,etiquetas_distintas,MetaProducto,ODS,11,11. Ciudades y comunidades sostenibles
6,etiquetas_distintas,MetaProducto,ODS,04,04. Educación de calidad
7,etiquetas_distintas,MetaProducto,ODS_META,01.01,"01.01. Para 2030, erradicar la pobreza extrema..."
8,etiquetas_distintas,MetaProducto,ODS_META,01.01,"01.01. De aquí a 2030, erradicar para todas la..."


,numero_metas
indicadores_por_meta,
1,409
2,14
3,2


### Resultados del diagnóstico y de la limpieza
- Corte analizado: **2026-06-30**; 5 tablas de datos; diccionario oficial presente.
- Celdas nulas en las tablas de datos antes de limpiar: **0**; filas duplicadas exactas: **0**.
- Celdas de texto normalizadas: **25**. Los códigos se tipificaron de forma consistente.
- Valores de avance fuera de [0, 1]: **0**. No se recortaron ni imputaron valores.
- Se conservaron todas las filas y se verificaron claves, fechas y relaciones jerárquicas.
- El detalle de las variantes de etiquetas está en la tabla de incidencias anterior; su redacción se conserva.

Estas comprobaciones describen integridad técnica del archivo; no certifican la exactitud sustantiva de lo reportado por las entidades ni sustituyen las fichas metodológicas de los indicadores.


### Interpretación de los controles del corte seleccionado

El resumen siguiente solo se genera después de ejecutar las comprobaciones anteriores. Distingo conservar filas, resolver representación y encontrar incidencias documentales: una preparación reproducible no exige ocultar las diferencias que siguen presentes en la fuente.



### Resultado calculado de conservación y relaciones

Calculo esta lectura con las variables de la ejecución, sin fijar conteos ni conclusiones de otro corte.


In [16]:
_relaciones_prosa = sum(int(r["registros_sin_padre"]) for r in control_relaciones)
_multiplicidad_prosa = limpias["Indicadores"].groupby("PLAN_META_PRODUCTO_CODIGO").size().value_counts().sort_index()
_detalle_multiplicidad = "; ".join(f"{int(n)} metas con {int(k)} indicador(es)" for k,n in _multiplicidad_prosa.items()) or "sin grupos observados"
display(Markdown(f"En **{fecha_corte.isoformat()}** verifico las claves y encuentro **{_relaciones_prosa} registros sin padre** en los cruces examinados. La relación meta–indicador presenta {_detalle_multiplicidad}. Registro **{len(incidencias)} incidencias** de etiquetas o rango en la tabla de control; conservarlas no implica corregir su significado."))


En **2026-06-30** verifico las claves y encuentro **0 registros sin padre** en los cruces examinados. La relación meta–indicador presenta 409 metas con 1 indicador(es); 14 metas con 2 indicador(es); 2 metas con 3 indicador(es). Registro **9 incidencias** de etiquetas o rango en la tabla de control; conservarlas no implica corregir su significado.

## 6. Conservación y escritura de las bases

Comparo las columnas originales con las preparadas y permito únicamente las transformaciones justificadas. Las correspondencias conservan el vínculo con el esquema recibido. En pivotes compruebo la recuperación de jerarquía y conservo los avances agregados.

Exporto un CSV por tabla disponible con delimitador `|`, punto decimal, UTF-8 con BOM y fechas ISO. Solo entrego diccionario si la fuente incluye esa tabla. Releo las bases y compruebo equivalencia con las tablas en memoria. El manifiesto y las huellas se muestran dentro del notebook.

Al importar declaro los identificadores como texto para preservar su representación. La existencia de un archivo de salida no sustituye la comprobación de su contenido.



In [7]:
# Comparación de todas las celdas originales, admitiendo solo las reglas documentadas.
# No basta con contar filas: también se verifican columnas, orden y contenido.
conservacion = []
for hoja, original in brutas.items():
    df = limpias[hoja]
    assert list(df.columns[:len(original.columns)]) == list(original.columns), hoja
    assert df.index.equals(original.index), f"Cambió el orden de filas: {hoja}"
    for col in original:
        esperado = original[col].map(espacio_limpio).map(
            lambda v: pd.NA if isinstance(v, str) and v == "" else v)
        if col.endswith("_CODIGO"):
            ancho = 2 if col in ["PLAN_NIVEL1_CODIGO", "PLAN_NIVEL2_CODIGO"] else None
            esperado = esperado.map(lambda v: codigo_texto(v, ancho)).astype("string")
        elif col.startswith("VIGENCIA_"):
            esperado = pd.to_numeric(esperado, errors="raise").astype("Int64")
        elif col == "FECHA_CORTE":
            esperado = pd.to_datetime(esperado, errors="raise")
        elif col in AVANCES:
            esperado = esperado.map(lambda v: float(v[:-1].strip().replace(",", "."))/100
                                     if isinstance(v, str) and v.endswith("%") else v)
            esperado = pd.to_numeric(esperado, errors="raise").astype("float64")
            if escalas_avance[hoja][col] == "porcentaje_0_100":
                esperado = esperado / 100
        else:
            esperado = esperado.astype("string")
        pd.testing.assert_series_equal(df[col], esperado, check_dtype=False, check_exact=True)
    conservacion.append({"hoja": hoja, "filas_conservadas": len(df),
                         "columnas_originales_verificadas": len(original.columns),
                         "resultado": "Contenido conservado según las reglas documentadas"})
display(pd.DataFrame(conservacion))


,hoja,filas_conservadas,columnas_originales_verificadas,resultado
0,PlanDesarrollo,1,10,Contenido conservado según las reglas document...
1,ObjetivoEstratégico,5,10,Contenido conservado según las reglas document...
2,Programa,39,12,Contenido conservado según las reglas document...
3,MetaProducto,425,16,Contenido conservado según las reglas document...
4,Indicadores,443,16,Contenido conservado según las reglas document...
5,DiccionarioDeDatos,16,4,Contenido conservado según las reglas document...


### Escritura y lectura de vuelta

En esta celda genero los CSV del esquema reconocido, los releo con sus tipos y comparo sus valores con la base en memoria. También muestro el manifiesto y una lectura de ejemplo para comprobar que los códigos conservan sus ceros iniciales.


In [8]:
import csv
DESTINO.mkdir(parents=True, exist_ok=True)
esquema = []
exportados = []
for hoja, df in limpias.items():
    ruta = DESTINO / SALIDAS[hoja]
    df.to_csv(ruta, index=False, encoding="utf-8-sig", sep="|", decimal=".", date_format="%Y-%m-%d", na_rep="", quoting=csv.QUOTE_ALL)
    tipos = {c: "string" for c in df if str(df[c].dtype) == "string"}
    fechas = [c for c in df if pd.api.types.is_datetime64_any_dtype(df[c])]
    vuelta = pd.read_csv(ruta, encoding="utf-8-sig", sep="|", dtype=tipos, parse_dates=fechas,
                         keep_default_na=False, na_values=[""], float_precision="round_trip")
    for col in df:
        if str(df[col].dtype) == "Int64":
            vuelta[col] = vuelta[col].astype("Int64")
        esquema.append({"archivo_csv": ruta.name, "hoja_origen": hoja, "columna": col,
                        "tipo_silver": str(df[col].dtype), "derivada": col not in brutas[hoja],
                        "unidad": "fraccion; 1 equivale a 100 %" if col in AVANCES else "",
                        "regla": "dos digitos" if col in ["PLAN_NIVEL1_CODIGO", "PLAN_NIVEL2_CODIGO", "ODS_CODIGO"] else ""})
    pd.testing.assert_frame_equal(df.reset_index(drop=True), vuelta, check_dtype=False, check_exact=True)
    exportados.append({"hoja": hoja, "archivo": ruta.name, "filas": len(df), "columnas": len(df.columns),
                       "sha256": hashlib.sha256(ruta.read_bytes()).hexdigest()})
manifiesto = {"fuente": fuente.relative_to(RAIZ).as_posix(), "sha256_fuente": hash_original,
              "fecha_corte": fecha_corte.isoformat(), "ejecucion_utc": datetime.now(timezone.utc).isoformat(),
              "versiones": {"python": platform.python_version(), "pandas": pd.__version__, "openpyxl": openpyxl.__version__},
              "csv": {"separador": "|", "decimal": ".", "encoding": "utf-8-sig", "nulos": "campo vacío"},
              "archivos": exportados, "esquema": esquema, "incidencias": len(incidencias),
              "version_esquema": version_esquema, "correspondencias": correspondencias.to_dict("records"),
              "politica": "Sin imputar, eliminar duplicados, recortar avances ni recalcular agregados"}
display(Markdown("**Trazabilidad de la ejecución**"))
display(manifiesto)
assert hashlib.sha256(fuente.read_bytes()).hexdigest() == hash_original
display(pd.DataFrame(exportados).drop(columns="sha256"))
print("CSV guardados y verificados en:", DESTINO)

# Ejemplo para usos posteriores: impedir que pandas transforme 01 en 1.
ejemplo = pd.read_csv(DESTINO / "avance_programa.csv", encoding="utf-8-sig", sep="|",
                     dtype={"PLAN_CODIGO": "string", "PLAN_NIVEL1_CODIGO": "string", "PLAN_NIVEL2_CODIGO": "string"},
                     parse_dates=["FECHA_CORTE"])
display(ejemplo.head(2))

# Control explícito de los casos revisados; se muestra lo realmente leído del CSV.
revision = pd.read_csv(DESTINO / "avance_indicadores.csv", sep="|", encoding="utf-8-sig",
                      dtype="string", keep_default_na=False)
metas_revisadas = ["2101", "2107", "2124", "2154"]
campos_revision = ["PLAN_META_PRODUCTO_CODIGO", "PLAN_INDICADOR_CODIGO", "PLAN_INDICADOR_NOMBRE",
                   "VIGENCIA_CORTE", "FECHA_CORTE", *AVANCES]
casos = revision.loc[revision.PLAN_META_PRODUCTO_CODIGO.isin(metas_revisadas), campos_revision]
display(casos)
# El control general anterior ya verifica todas las filas; esta vista facilita la revisión manual.


C:\Users\USER\AppData\Local\Temp\ipykernel_30644\2813915306.py:10: Pandas4Warning: The 'float_precision' argument is deprecated. Use the default float precision instead.
  vuelta = pd.read_csv(ruta, encoding="utf-8-sig", sep="|", dtype=tipos, parse_dates=fechas,
C:\Users\USER\AppData\Local\Temp\ipykernel_30644\2813915306.py:10: Pandas4Warning: The 'float_precision' argument is deprecated. Use the default float precision instead.
  vuelta = pd.read_csv(ruta, encoding="utf-8-sig", sep="|", dtype=tipos, parse_dates=fechas,
C:\Users\USER\AppData\Local\Temp\ipykernel_30644\2813915306.py:10: Pandas4Warning: The 'float_precision' argument is deprecated. Use the default float precision instead.
  vuelta = pd.read_csv(ruta, encoding="utf-8-sig", sep="|", dtype=tipos, parse_dates=fechas,
C:\Users\USER\AppData\Local\Temp\ipykernel_30644\2813915306.py:10: Pandas4Warning: The 'float_precision' argument is deprecated. Use the default float precision instead.
  vuelta = pd.read_csv(ruta, encoding="ut

**Trazabilidad de la ejecución**

{'fuente': 'Capa Bronce/2026-06-30/avance_plan/01_avancefisico20260630_d6da6ba83a.xlsx',
 'sha256_fuente': 'c7464dc5bfdbc96d5bd39340c4b0958d38b4394727670cf4bfd54719cae543a0',
 'fecha_corte': '2026-06-30',
 'ejecucion_utc': '2026-10-29T21:13:57.848935+00:00',
 'versiones': {'python': '3.14.7', 'pandas': '3.1.0', 'openpyxl': '3.1.5'},
 'csv': {'separador': '|',
  'decimal': '.',
  'encoding': 'utf-8-sig',
  'nulos': 'campo vacío'},
 'archivos': [{'hoja': 'PlanDesarrollo',
   'archivo': 'avance_plan.csv',
   'filas': 1,
   'columnas': 10,
   'sha256': '72f26def93c2d004f34063dbd876847308e58da09e3e5f6519aa356d58ab0b62'},
  {'hoja': 'ObjetivoEstratégico',
   'archivo': 'avance_objetivo_estrategico.csv',
   'filas': 5,
   'columnas': 10,
   'sha256': 'bba3e6bc6d011b28dd0b2f07ca6e4c2aec78052b91fd19b4eeaf76d84919ddc1'},
  {'hoja': 'Programa',
   'archivo': 'avance_programa.csv',
   'filas': 39,
   'columnas': 12,
   'sha256': '20acd8a0a2223f7ab0b57e6faaa33aca19398bf520434ca3fbfc79b95dcc2806'},


,hoja,archivo,filas,columnas
0,PlanDesarrollo,avance_plan.csv,1,10
1,ObjetivoEstratégico,avance_objetivo_estrategico.csv,5,10
2,Programa,avance_programa.csv,39,12
3,MetaProducto,avance_meta_producto.csv,425,18
4,Indicadores,avance_indicadores.csv,443,16
5,DiccionarioDeDatos,diccionario_avance_plan.csv,16,4


CSV guardados y verificados en: C:\Users\USER\Desktop\Scraping_proyecto_de_grado\Capa Silver\2026-06-30\avance_plan


,PLAN_CODIGO,PLAN_NOMBRE,PLAN_SIGLA,PLAN_NIVEL1_CODIGO,PLAN_NIVEL1_NOMBRE,PLAN_NIVEL2_CODIGO,PLAN_NIVEL2_NOMBRE,VIGENCIA_CORTE,FECHA_CORTE,AVANCE_VIGENCIA,AVANCE_TRANSCURRIDO,AVANCE_PLAN
0,17,Bogotá Camina Segura,BCS,01,Bogotá avanza en su seguridad,01,Diálogo social y cultura ciudadana para la con...,2026,2026-06-30,0.530220,0.805617,0.592122
1,17,Bogotá Camina Segura,BCS,01,Bogotá avanza en su seguridad,02,Cero tolerancia a las violencias contra las mu...,2026,2026-06-30,0.869491,0.970348,0.747789


,PLAN_META_PRODUCTO_CODIGO,PLAN_INDICADOR_CODIGO,PLAN_INDICADOR_NOMBRE,VIGENCIA_CORTE,FECHA_CORTE,AVANCE_VIGENCIA,AVANCE_TRANSCURRIDO,AVANCE_PLAN
176,2101,4023,Estrategia que fortalezca a maestros y maestra...,2026,2026-06-30,0.15,0.7166666666666667,0.5375
182,2107,4029,Número de jóvenes que se benefician con progra...,2026,2026-06-30,1.0,1.0,0.63659375
199,2124,4046,Estrategia para el fortalecimiento de la Agenc...,2026,2026-06-30,0.0,1.0,1.0
229,2154,4076,Espacios redes plataformas y encuentros estrat...,2026,2026-06-30,0.04912000000000001,0.6583564864864865,0.5289457142857142


## 7. Alcance de la actualización

La preparación utiliza reconocimiento de perfil, homologación trazable, conversión documentada y comprobación de jerarquías. Los resúmenes anteriores se recalculan para el corte seleccionado: no fijo cardinalidades, ausencia de nulos ni relaciones uno a uno como conclusiones permanentes.

Una estructura nueva requiere revisar su adaptador y documentar la decisión antes de aceptarla. La validación histórica demuestra lo probado en esas fuentes; no certifica planes o archivos futuros. La integridad técnica tampoco demuestra la exactitud sustantiva de los avances reportados.



---

<details>
<summary>Anexo histórico: evidencias y redacción de las revisiones anteriores — no describe el corte seleccionado</summary>

Conservo estos textos como registro de revisiones puntuales, realizadas sobre las fuentes y fechas que cada apartado identifica. Sus conteos, ejemplos, resultados y límites pertenecen a esas versiones. No son resultados recalculados ni condiciones satisfechas para una nueva ejecución. La metodología operativa y los informes del corte seleccionado están en el cuerpo principal. Las revisiones históricas no se amplían automáticamente.

# Exploración y preparación reproducible de Avance del Plan

## 1. Qué necesito resolver antes de limpiar

En este notebook preparo las tablas físicas del plan sin alterar sus originales. Primero identifico el esquema, la unidad de observación y la escala de sus avances; después justifico y aplico transformaciones de representación y verifico su conservación. No trato plan, objetivo, programa, meta e indicador como filas equivalentes ni sumo sus avances entre niveles.

La revisión de otros cortes cambió mi diagnóstico inicial: no basta con que un Excel abra correctamente. Desde 2024 encontré encabezados diferentes, valores porcentuales en escala 0–100, ausencia de diccionario y, en septiembre de 2026, pivotes con totales y detalle por entidad. Por eso incorporé adaptadores explícitos y mantuve la lectura original en `brutas_fuente`, junto con una tabla de correspondencias.

**Periodo de la revisión.** Realicé la revisión histórica el 29 de octubre de 2026 con las versiones que el catálogo de la SDP ofrecía ese día. Distingo el corte de la publicación, la fecha de la observación y la versión del archivo. Las cifras históricas de este texto son resultados de esa revisión; las tablas ejecutables se recalculan para la fuente seleccionada. No atribuyo las conclusiones de un archivo a periodos futuros sin repetir los controles.

La configuración selecciona un corte local. Si no fijo `CORTE`, uso el último con Excel en ambas fuentes; si fijo una fecha, reviso esa fuente concreta. Para cambiar la ubicación del proyecto mantengo `rutas.py` junto a los notebooks. La instalación siguiente solo hace falta si todavía no dispongo de las dependencias.


---

## 3. Reconocimiento del esquema y de la escala

En el siguiente bloque identifico uno de los esquemas que contrasté en las fuentes históricas. Conservo los DataFrames originales y explicito los cambios de nombre antes de limpiar. No infiero que un número pequeño sea una fracción y uno grande un porcentaje: uso el esquema documentado y el formato de las celdas. Para marzo y junio de 2025 reconozco expresamente la versión en escala 0–100, aunque ya utilice encabezados en mayúsculas.

En los pivotes de septiembre de 2026 separo el `Total general` de las observaciones de cada nivel y reconstruyo los padres desde el detalle por entidad e indicador. Utilizo los avances del pivote tal como se publicaron; no los recalculo con medias simples. Conservo en memoria los totales y los pies sin clave como auxiliares para poder revisar esta decisión.

Si una versión no aporta diccionario, no fabrico uno oficial. Si no incluye fecha dentro del libro, como la versión de pivotes, documento que el corte procede de la publicación y de la carpeta seleccionada. Ante un esquema o una escala no reconocidos detengo el proceso. También compruebo que el archivo identifique un solo plan.


---

### Qué observé en el corte local de junio de 2026

Encontré 1 registro del plan, 5 objetivos, 39 programas, 425 metas producto y 443 indicadores, además de 16 registros en el diccionario. No observé fórmulas ni errores en las tablas consumidas. Esta diferencia de cardinalidades me llevó a conservar los niveles separados y a comprobar sus relaciones mediante claves.

En las columnas de avance encontré formato `0.00%` y números fraccionarios. Interpreté `0,75` como 75 % y conservé ese valor interno. Esta conclusión corresponde a esta versión: los archivos de 2024 y del primer semestre de 2025 requieren dividir los porcentajes entre cien. El adaptador distingue ambos casos antes de la transformación.


---

## 4. Exploración de calidad antes de transformar

Una vez homologado el esquema reviso vacíos, tipos, valores distintos, duplicados y claves. Identifico textos que cambiarían al normalizar espacios y Unicode y examino los avances por nivel, mediante mínimo, cuartiles, mediana, máximo y frecuencia de ceros.

Para comparar distribuciones expreso la vista diagnóstica de los avances en escala 0–1 según la escala ya reconocida; no modifico `brutas` en esta exploración. Así evito clasificar un 74,78 % histórico como un valor anómalo por estar por encima de uno. La limpieza posterior realiza y verifica la conversión sobre una copia.

Uso los resultados para decidir si hay un problema de representación o una incidencia que requiere revisión. No elimino una observación por ser extrema ni convierto un cero en un faltante. El perfil completo queda en memoria y la vista resumida muestra las columnas con diferencias relevantes.


---

### Cómo interpreté los hallazgos del corte local

No encontré filas vacías, duplicados exactos ni claves nulas o repetidas en junio de 2026. Las cinco tablas analíticas no tenían faltantes. Los 15 vacíos del diccionario estaban en `Observación`; los interpreté como ausencia de una nota documental y no imputé mediciones.

Localicé 25 celdas que cambiaban al normalizar espacios o Unicode: 2 nombres de meta en `MetaProducto`, 18 etiquetas ODS, 2 etiquetas de meta ODS, 2 nombres de meta en `Indicadores` y 1 nombre de indicador. Por eso adopté una limpieza textual acotada, sin reescribir redacción, acentos o mayúsculas.

También observé el mismo programa representado como `1` y `01`. Dado que cumple una función de identificación, lo homogeneicé como texto de dos dígitos. Los ceros de avance se conservaron como valores reportados. No extrapolé la ausencia de vacíos de junio a otros cortes.


---

## 5. Reglas que adopté y por qué

| Hallazgo o condición | Decisión que implemento | Límite que mantengo |
|---|---|---|
| Códigos de objetivo y programa con representaciones diferentes | Los expreso como texto de dos dígitos | No aplico una longitud única a todos los códigos |
| Espacios y Unicode no uniformes | Normalizo representación textual | No cambio el significado ni elijo una redacción oficial |
| Avances históricos en escala 0–100 | Los divido entre cien cuando el perfil reconocido lo documenta | No decido la escala a partir del máximo observado |
| Avances fraccionarios con formato porcentual | Conservo el valor interno | No divido una segunda vez |
| Fechas explícitas, a veces con estado entre paréntesis | Extraigo la fecha DD/MM/AAAA y valido corte y vigencia | No adivino formatos ambiguos |
| Etiquetas ODS disponibles | Derivo sus códigos manteniendo las etiquetas | Si la fuente no aporta ODS, no los invento |
| Totales generales en pivotes | Los separo de las observaciones y los conservo en auxiliares | No los cuento como un objetivo, programa, meta o indicador adicional |
| Vacíos, variantes de nombre y valores extremos | Los diagnostico y preservo | No imputo, deduplico ni recorto sin una regla sustentada |

En la celda siguiente aplico estas decisiones a una copia. El archivo de Bronce y los DataFrames de lectura fuente siguen disponibles para contrastar lo realizado.


---

## 6. Cómo compruebo el efecto de las decisiones

Repito el perfil después de limpiar y verifico número de observaciones, claves, fecha y relaciones entre niveles. Contrasto los avances con su entrada, admitiendo únicamente la conversión de escala documentada. Cuando la fuente contiene inicio y fin del plan, también compruebo sus límites temporales; si no los aporta, no los imputo.

La normalización de códigos debe permitir los cruces sin multiplicar filas ni dejar hijos sin padre. Registro las variantes de nombre asociadas al mismo código como incidencias. No elijo una etiqueta por frecuencia ni sustituyo avances agregados por una media simple de sus hijos.


---

### Qué pude verificar después de preparar junio de 2026

Conservé las filas de las seis hojas y no encontré relaciones sin referencia. Añadí los dos códigos ODS a `MetaProducto`, que pasó de 16 a 18 columnas. La relación meta–indicador mostró 409 metas con un indicador, 14 con dos y 2 con tres; esta evidencia me impidió imponer una relación uno a uno.

Conservé las variantes de nombres de programa y de etiquetas ODS porque la coincidencia del código no demuestra cuál redacción debe reemplazar a las demás. La preparación resolvió diferencias de representación y dejó visibles las incidencias documentales.


---

## 7. Conservación y escritura de las bases

Comparo cada columna del esquema homologado con su versión preparada y permito solo las transformaciones ya justificadas. Para auditar nombres o representaciones previas puedo volver a `brutas_fuente` y `correspondencias`. En el formato de pivotes, la recuperación de jerarquía se verifica mediante cruces uno a uno y los avances agregados se conservan.

Exporto un CSV por tabla disponible con delimitador `|`, punto decimal, UTF-8 con BOM y fechas ISO. Solo entrego CSV del diccionario cuando la fuente incluye esa hoja. Releo las bases y exijo equivalencia con las tablas en memoria. El manifiesto, la versión del esquema, las correspondencias y las huellas se muestran dentro del notebook, sin escribir JSON auxiliares.

Al importar declaro los códigos como texto. CSV no conserva tipos y una lectura automática podría convertir `01` nuevamente en `1`. El ejemplo final muestra una lectura compatible.


---

## 8. Conclusión sobre la automatización de Avance del Plan

La revisión histórica me permitió comprobar que la limpieza inicial era adecuada para junio de 2026, pero no para todos los archivos anteriores ni para el nuevo formato de septiembre. Corregí esas dependencias mediante reconocimiento de versión, homologación trazable de nombres, conversión de escala documentada y recuperación de la jerarquía desde el detalle de los pivotes.

La actualización rutinaria queda automatizada para los perfiles probados. Si aparece una estructura diferente, mantengo una detención explícita en lugar de asumir equivalencias. No presento esta validación de BCS como prueba de compatibilidad con las metas sectoriales de UNCSAB ni con cualquier plan futuro.

Mi resultado es una preparación reproducible y técnicamente comprobada de las tablas publicadas. La conservación y los cruces no certifican la exactitud sustantiva de los avances reportados por las entidades.


---

## 9. Auditoría independiente de conservación - corte 2026-06-30

Revisión realizada el 28 de octubre de 2026 sobre los archivos existentes, sin regenerar los CSV. El Excel se leyó directamente con `openpyxl` y los CSV con `csv.reader` (delimitador `|`, lectura estricta), independientemente de pandas y de las funciones de limpieza del notebook.

| Tabla | Filas Excel/CSV | Columnas originales/CSV | Celdas originales verificadas | Diferencias indebidas |
|---|---:|---:|---:|---:|
| Plan | 1/1 | 10/10 | 10 | 0 |
| Objetivos estratégicos | 5/5 | 10/10 | 50 | 0 |
| Programas | 39/39 | 12/12 | 468 | 0 |
| Metas producto | 425/425 | 16/18 | 6.800 | 0 |
| Indicadores | 443/443 | 16/16 | 7.088 | 0 |
| Diccionario | 16/16 | 4/4 | 64 | 0 |

**Resultado:** 929 filas y 14.480 celdas originales comparadas. Se verificaron encabezados, orden, anchura de todos los registros CSV, contenido completo de textos, signos especiales, 2.739 avances con igualdad numérica exacta, 913 fechas sin componentes horarios omitidos, 4.043 códigos y los 15 vacíos originales del diccionario. Las letras, cifras y signos de los textos se conservaron; las 25 diferencias textuales corresponden a espacios o normalización Unicode. Las 529 diferencias de representación de códigos conservan su valor identificador. Los dos campos ODS adicionales se comprobaron para las 425 metas.

Las metas 2101, 2107, 2124 y 2154 conservan respectivamente los indicadores 4023, 4029, 4046 y 4076 y todos los campos reportados como ausentes. La auditoría no modificó Bronce ni los CSV; se comprobaron sus huellas antes y después.

Esta es una evidencia puntual del corte auditado, no una certificación de futuros archivos. Los controles ejecutables de las secciones anteriores se aplican en cada nueva ejecución. La conservación se refiere a datos tabulares; CSV no reproduce estilos ni formatos visuales de Excel, que permanecen en Bronce.

SHA-256 del Excel original: `c7464dc5bfdbc96d5bd39340c4b0958d38b4394727670cf4bfd54719cae543a0`.

---

## 10. Validación histórica de los adaptadores

Probé el procesamiento, los controles de conservación y la relectura de los CSV en una copia fuera de la carpeta de entrega. Los nueve cortes numéricos de BCS disponibles pasaron las comprobaciones de jerarquía sin registros sin padre.

| Corte | Perfil reconocido | Metas | Indicadores | Diccionario en la fuente |
|---|---|---:|---:|---|
| 2024-09-30 | tabular_historico_0_100 | 425 | 439 | No |
| 2024-12-31 | tabular_historico_0_100 | 425 | 439 | No |
| 2025-03-31 | tabular_2025_0_100 | 425 | 439 | Sí |
| 2025-06-30 | tabular_2025_0_100 | 425 | 439 | Sí |
| 2025-09-30 | tabular_fracciones | 425 | 440 | Sí |
| 2025-12-31 | tabular_fracciones | 425 | 441 | Sí |
| 2026-03-31 | tabular_fracciones | 425 | 441 | Sí |
| 2026-06-30 | tabular_fracciones | 425 | 443 | Sí |
| 2026-09-30 | pivotes_y_catalogo_2026 | 425 | 443 | No |

En septiembre de 2026 separé una fila de `Total general` en cada uno de los cuatro pivotes con códigos. En la hoja de detalle encontré 517 registros con clave y cuatro filas finales sin identidad de plan: tres vacías y una con valores auxiliares de ponderación. Las conservé aparte; no las conté como observaciones institucionales. Esto explica por qué el número de filas físicas del libro no coincide con el número de entidades–indicadores.

Las fuentes enlazadas y la distinción entre planes están en la revisión del notebook 01. Para repetir una prueba selecciono el corte correspondiente; no fijo estos conteos como condiciones del código.


</details>
